## LiteLLM
LiteLLM is an open-source Python library that provides a unified interface to interact with multiple LLM providers, such as OpenAI, Gemini, Anthropic, and Ollama. 

Let's understand some important concepts of LiteLLM, including Abstraction Layer, Router, Prompt Caching, and Model-to-Model Interaction (Multi-Model Communication), which help us build flexible, efficient, and scalable AI applications by simplifying model integration, managing requests, reducing repeated processing, and enabling multiple LLMs to collaborate on complex tasks.

### LLM Abstraction Layer

An Abstraction Layer hides the internal complexity of different LLM providers and gives developers a common interface to interact with them.

In [10]:
from litellm import completion
from IPython.display import display, Markdown

response = completion(model="ollama/qwen2.5-coder:1.5b", messages=[
    {"role": "user", "content": "Tell a joke"}
])

ans = response.choices[0].message.content

display(Markdown(ans))


Sure, here's a joke for you:

Why don't scientists trust atoms?

Because they make up everything.

In [11]:
print(f"Input token: {response.usage.prompt_tokens}")
print(f"Output token: {response.usage.completion_tokens}")
print(f"Total token: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params['response_cost']*100:.4f} cents")

Input token: 36
Output token: 23
Total token: 59
Total cost: 0.0000 cents


#### Change model with Gemini

In [7]:
from litellm import completion
from IPython.display import display, Markdown

response = completion(model="gemini/gemini-3.1-flash-lite", messages=[
    {"role": "user", "content": "Tell a joke"}
])

ans = response.choices[0].message.content

display(Markdown(ans))


Why don't scientists trust atoms?

Because they make up everything!

In [9]:
print(f"Input token: {response.usage.prompt_tokens}")
print(f"Output token: {response.usage.completion_tokens}")
print(f"Total token: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params['response_cost']*100:.4f} cents")

Input token: 4
Output token: 15
Total token: 19
Total cost: 0.0024 cents


### LLM Router

An LLM Router decides which LLM model should handle a particular request.

Imagine your application has three models:

Gemini: General questions.

Ollama: Coding tasks.

OpenAI: Complex reasoning.

Instead of manually selecting a model every time, a router can select one based on the request.

In [24]:
from litellm import completion
from IPython.display import display, Markdown

def llm_router(user_query):
    
    if "code" in user_query.lower() or "python" in user_query.lower():
        model = "ollama/qwen2.5-coder:1.5b"
    else:
        model = "gemini/gemini-3.1-flash-lite"

    response = completion(
        model = model,
        messages=[{"role": "user", "content": user_query}]
    )

    answer = response.choices[0].message.content

    display(Markdown(answer))

llm_router("write a python function to calculate the factorial of a number")


Certainly! Below is a Python function that calculates the factorial of a given number:

```python
def factorial(n):
    # Base case: factorial of 0 or 1 is 1
    if n == 0 or n == 1:
        return 1
    
    # Recursive case: n * factorial of (n-1)
    else:
        return n * factorial(n-1)
```

### Explanation:
- **Base Case**: The function first checks if the input number `n` is either 0 or 1. In these cases, the factorial is defined as 1 because the factorial of 0 and 1 is 1.
- **Recursive Case**: If `n` is greater than 1, the function calls itself with the argument `n-1` and multiplies the result by `n`. This recursive call effectively multiplies `n` by the factorial of all previous numbers leading up to `n`.

### Example Usage:
You can call the `factorial` function with any non-negative integer to get its factorial. For example:

```python
print(factorial(5))  # Output: 120
print(factorial(0))  # Output: 1
print(factorial(3))  # Output: 6
```

This function handles all non-negative integers and correctly calculates their factorial.

#### Example using LiteLLM Router

In [10]:
import os
from litellm import Router
from IPython.display import display, Markdown

model_list = [
    {
        "model_name": "gemini-model",
        "litellm_params": {
            "model": "gemini/gemini-3.1-flash-lite",
            "api_key": os.getenv("GOOGLE_API_KEY")
        }
    },
    {
        "model_name": "local-model",
        "litellm_params": {
            "model": "ollama/qwen2.5-coder:1.5b",
            "api_base": "http://localhost:11434"
        }
    }
]

router = Router(model_list=model_list)

response = router.completion(
    model="gemini-model",
    messages=[
        {"role": "user", "content": "Tell me a fact"}
    ]
)

answer = response.choices[0].message.content

display(Markdown(answer))

A **cloud** can weigh more than a million pounds. 

Even though they look fluffy and light, a typical cumulus cloud (the puffy, white ones) contains so much water vapor that it can weigh about 1.1 million pounds—roughly the weight of 100 elephants! They stay afloat because the air beneath them is even denser than the cloud itself.